In [1]:
import gc
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc, torch
import torchvision.transforms as transforms
from IPython.display import display
from torch.utils.data import DataLoader

from codes.sprint.models import *
from codes.sprint.training import *
from codes.sprint.data import *
from codes.sprint.utils import *


In [3]:
MODEL_REGISTRY = {
    "A2": Model_SchemeA2,
    "C2": Model_SchemeC2,
    "RNA_Only0701": Model_RNA_Only,
    "HE_Only": Model_SchemeA2_HE_Only
}

# ---- Image resolution auto-selection ----
# Set MANUAL_SCHEME = "A" or "C" to force, or None for auto-detect.
# Auto-detect rule (based on spleen/breast/tonsil/brain/msi experiments):
#   patch >= 128 px -> Scheme A (high-res, 256 spatial tokens)
#   patch <  128 px -> Scheme C (low-res, 4 global-pool tokens)
MANUAL_SCHEME = None
# -----------------------------------------

selected_scheme = select_image_processor_scheme(
    h5ad_path="datas/tonsil/Tonsil_1_Final.h5ad",
    manual_scheme=MANUAL_SCHEME
)
selected_model_key = f"{selected_scheme}2"
print(f"[Auto] Image resolution -> Scheme {selected_scheme}, model: {selected_model_key}")

MAIN_MODELS = [
    (MODEL_REGISTRY[selected_model_key], selected_model_key)
]

ABLATION_MODELS = [
    # (MODEL_REGISTRY["RNA_Only0701"], "RNA_Only0701"),
    (MODEL_REGISTRY["HE_Only"], "HE_Only")
]


[Auto] Image resolution -> Scheme A, model: A2


In [4]:
path_train = "datas/tonsil/Tonsil_1_Final.h5ad"
path_val = "datas/tonsil/Tonsil_2_Final.h5ad"
if not (os.path.exists(path_train) and os.path.exists(path_val)):
    raise FileNotFoundError("Tonsil h5ad files not found")
ad1 = sc.read_h5ad(path_train, backed="r")
ad2 = sc.read_h5ad(path_val, backed="r")
p1 = [str(x) for x in list(ad1.uns["protein_names"])] if "protein_names" in ad1.uns else [str(i) for i in range(ad1.obsm["protein_expression_log"].shape[1])]
p2 = [str(x) for x in list(ad2.uns["protein_names"])] if "protein_names" in ad2.uns else [str(i) for i in range(ad2.obsm["protein_expression_log"].shape[1])]
common_proteins = sorted(set(p1).intersection(p2))
del ad1, ad2
gc.collect()

1208

In [5]:
train_dataset = TonsilMultimodalDataset(path_train, target_proteins=common_proteins)
val_dataset = TonsilMultimodalDataset(path_val, target_proteins=common_proteins)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
NUM_GENES = train_dataset.rna_data.shape[1]
NUM_PROTEINS = train_dataset.protein_data.shape[1]
print(f"Final Config: {NUM_GENES} genes, {NUM_PROTEINS} proteins")

Loading data from: datas/tonsil/Tonsil_1_Final.h5ad ...
Filtering proteins → 29 proteins.
Loading data from: datas/tonsil/Tonsil_2_Final.h5ad ...
Filtering proteins → 29 proteins.
Final Config: 18085 genes, 29 proteins


In [6]:
model_kwargs = {"num_genes": NUM_GENES}
models_to_train = MAIN_MODELS

In [6]:
all_histories = {}

In [ ]:
for i, (model_cls, name) in enumerate(models_to_train):
    print(f"\n[{i + 1}/{len(models_to_train)}] Training {name}")
    all_histories[name] = train_engine(model_cls, name, train_loader, val_loader, NUM_PROTEINS, device, epochs=30, model_kwargs=model_kwargs)

In [7]:
ablation_models = ABLATION_MODELS

In [9]:
for i, (model_cls, name) in enumerate(ablation_models):
    print(f"\n[{i + 1}/{len(ablation_models)}] Ablation {name}")
    all_histories[name] = train_engine(model_cls, name, train_loader, val_loader, NUM_PROTEINS, device, lr=1e-4, epochs=30, model_kwargs=model_kwargs)



[1/1] Ablation HE_Only


[HE_Only] Ep 1/30:   0%|          | 0/132 [00:00<?, ?it/s]

HE_Only epoch 1: loss=4.8575, val_pcc=0.1106, val_rmse=0.7566


[HE_Only] Ep 2/30:   0%|          | 0/132 [00:00<?, ?it/s]

HE_Only epoch 2: loss=0.4443, val_pcc=0.1692, val_rmse=0.7986


...

HE_Only epoch 28: loss=0.0510, val_pcc=0.2696, val_rmse=0.7037


[HE_Only] Ep 29/30:   0%|          | 0/132 [00:00<?, ?it/s]

HE_Only epoch 29: loss=0.0454, val_pcc=0.2804, val_rmse=0.7157


[HE_Only] Ep 30/30:   0%|          | 0/132 [00:00<?, ?it/s]

HE_Only epoch 30: loss=0.0417, val_pcc=0.2715, val_rmse=0.6667
